# S04E04 — Filesystem (Notatki Natana)

Zadanie: logiczne uporządkowanie notatek Natana w wirtualnym filesystemie.

Wymagana struktura:
- `/miasta` — pliki JSON z towarami potrzebnymi miastom
- `/osoby` — pliki markdown z linkami do miast (imię/nazwisko → miasto)
- `/towary` — pliki z linkami do miast oferujących dany towar

Notatki Natana: https://hub.ag3nts.org/dane/natan_notes.zip

Podejście agentowe:
1. Pobierz i przeanalizuj notatki
2. Wyekstrahuj dane o miastach, osobach i towarach
3. Zbuduj strukturę filesystemu przez API

In [ ]:
import os, json, requests, zipfile, io
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")
API_KEY = os.getenv("AI_DEVS_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

VERIFY_URL = "https://hub.ag3nts.org/verify"
TASK = "filesystem"
NOTES_URL = "https://hub.ag3nts.org/dane/natan_notes.zip"

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [ ]:
# Pobierz notatki Natana
print("Pobieram notatki Natana...")
r = requests.get(NOTES_URL)
z = zipfile.ZipFile(io.BytesIO(r.content))

notes = {}
for name in z.namelist():
    content = z.read(name).decode("utf-8", errors="replace")
    notes[name] = content
    print(f"  {name}: {len(content)} znaków")

print(f"\nŁącznie: {len(notes)} plików")

In [ ]:
# Wyświetl treść notatek
for name, content in notes.items():
    print(f"\n{'='*50}")
    print(f"PLIK: {name}")
    print('='*50)
    print(content[:1000])

## Krok 1: Analiza notatek przez Claude

In [ ]:
# Połącz wszystkie notatki w jeden tekst do analizy
all_notes_text = "\n\n".join(
    f"=== Plik: {name} ===\n{content}"
    for name, content in notes.items()
)

ANALYSIS_PROMPT = """Przeanalizuj poniższe notatki Natana dotyczące handlu między miastami.

Wyekstrahuj i zwróć JSON z następującą strukturą:
{
  "miasta": {
    "nazwa_miasta": {
      "potrzeby": {"towar": ilosc, ...},
      "oferuje": ["towar1", "towar2", ...]
    }
  },
  "osoby": [
    {
      "imie_nazwisko": "Jan Kowalski",
      "miasto": "nazwa_miasta"
    }
  ]
}

WAŻNE:
- Nazwy miast w mianowniku (bez polskich znaków)
- Nazwy towarów w mianowniku liczby pojedynczej (bez polskich znaków)
- Ilości jako liczby (bez jednostek)
- Jeśli nie wiesz ilości, wpisz 0

NOTATKI:
"""

analysis_response = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=4096,
    messages=[{
        "role": "user",
        "content": ANALYSIS_PROMPT + all_notes_text
    }]
)

analysis_text = analysis_response.content[0].text
print(analysis_text[:3000])

In [ ]:
# Wyekstrahuj JSON z odpowiedzi
import re

json_match = re.search(r'```(?:json)?\s*({[\s\S]+?})\s*```', analysis_text)
if json_match:
    extracted_data = json.loads(json_match.group(1))
else:
    # Spróbuj bezpośrednio
    start = analysis_text.find('{')
    end = analysis_text.rfind('}') + 1
    extracted_data = json.loads(analysis_text[start:end])

print("Wyekstrahowane dane:")
print(json.dumps(extracted_data, indent=2, ensure_ascii=False))

## Krok 2: Budowanie struktury filesystemu przez API

In [ ]:
# Funkcja pomocnicza — wywołanie API filesystem
def fs_api(answer) -> dict:
    payload = {"apikey": API_KEY, "task": TASK, "answer": answer}
    r = requests.post(VERIFY_URL, json=payload)
    return r.json()

def fs_show(answer) -> dict:
    result = fs_api(answer)
    action_str = json.dumps(answer) if not isinstance(answer, list) else f"[batch: {len(answer)} operacji]"
    print(f">> {action_str[:100]}")
    print(f"<< {json.dumps(result, ensure_ascii=False)[:300]}")
    return result

# Sprawdź dostępne akcje
fs_show({"action": "help"})

In [ ]:
# Reset stanu (czyści wszystkie pliki)
fs_show({"action": "reset"})

In [ ]:
# Przygotuj batch operacji
def sanitize(text: str) -> str:
    """Usuń polskie znaki i zamień spacje na podkreślniki."""
    mapping = {
        'ą': 'a', 'ć': 'c', 'ę': 'e', 'ł': 'l', 'ń': 'n',
        'ó': 'o', 'ś': 's', 'ź': 'z', 'ż': 'z',
        'Ą': 'A', 'Ć': 'C', 'Ę': 'E', 'Ł': 'L', 'Ń': 'N',
        'Ó': 'O', 'Ś': 'S', 'Ź': 'Z', 'Ż': 'Z'
    }
    result = ''.join(mapping.get(c, c) for c in text)
    return result.replace(' ', '_')

operations = []

# Utwórz katalogi
for d in ["/miasta", "/osoby", "/towary"]:
    operations.append({"action": "createDir", "path": d})

# /miasta — pliki JSON z potrzebami miast
for miasto, dane in extracted_data["miasta"].items():
    nazwa_pliku = sanitize(miasto.lower())
    potrzeby = dane.get("potrzeby", {})
    # Sanitize klucze w potrzebach
    potrzeby_clean = {sanitize(k.lower()): v for k, v in potrzeby.items()}
    content = json.dumps(potrzeby_clean, ensure_ascii=False)
    operations.append({
        "action": "createFile",
        "path": f"/miasta/{nazwa_pliku}",
        "content": content
    })

# /osoby — pliki markdown z linkami do miast
for osoba in extracted_data["osoby"]:
    imie_nazwisko = osoba["imie_nazwisko"]
    miasto = osoba["miasto"]
    nazwa_pliku = sanitize(imie_nazwisko)
    miasto_sanitized = sanitize(miasto.lower())
    content = f"[{miasto}](/miasta/{miasto_sanitized})"
    operations.append({
        "action": "createFile",
        "path": f"/osoby/{nazwa_pliku}",
        "content": content
    })

# /towary — pliki z linkami do miast oferujących towar
towary_do_miast = {}
for miasto, dane in extracted_data["miasta"].items():
    for towar in dane.get("oferuje", []):
        towar_clean = sanitize(towar.lower())
        if towar_clean not in towary_do_miast:
            towary_do_miast[towar_clean] = []
        towary_do_miast[towar_clean].append(miasto)

for towar, miasta_list in towary_do_miast.items():
    miasto = miasta_list[0]  # Każdy towar jest z jednego miasta
    miasto_sanitized = sanitize(miasto.lower())
    content = f"[{miasto}](/miasta/{miasto_sanitized})"
    operations.append({
        "action": "createFile",
        "path": f"/towary/{towar}",
        "content": content
    })

print(f"Przygotowano {len(operations)} operacji:")
for op in operations:
    print(f"  {op['action']}: {op.get('path', '')}")

In [ ]:
# Wyślij wszystkie operacje w batch mode
print("Wysyłam operacje batch...")
batch_result = fs_show(operations)
print("\nWynik batch:")
print(json.dumps(batch_result, indent=2, ensure_ascii=False))

In [ ]:
# Weryfikuj stan filesystemu
for d in ["/miasta", "/osoby", "/towary"]:
    result = fs_api({"action": "listDir", "path": d})
    print(f"\n{d}:")
    print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Wyślij do weryfikacji
print("Wysyłam do weryfikacji...")
done_result = fs_show({"action": "done"})
print("\nWynik:")
print(json.dumps(done_result, indent=2, ensure_ascii=False))

# Szukaj flagi
result_str = json.dumps(done_result)
if "FLG" in result_str:
    print(f"\n{'*'*60}")
    print("FLAGA ZNALEZIONA!")
    print(result_str)
    print('*'*60)